# 02 – Analysis of the FinQA results

**Seminar paper:** Benchmarks for Financial LLM Agents. This notebook covers the empirical part (step 2 of 2).

Reads the raw answers from `results/*.jsonl` (produced by `01_run_experiments.ipynb`) and computes all results. No model calls; runs in seconds.

**Scoring**
- **Strict exact match**, close to FinBen's EmAcc: the number as the model wrote it, rounded to 2 decimals, must equal the exact gold result rounded to 2 decimals. No rescaling, so `14.46%` ≠ `0.14464`.
- **Tolerant numeric match:** within 1% of the exact result or of the rounded human-written answer; percent and decimal forms count as equal (`14%` = `0.14`).

The difference between the two shows how much a benchmark score depends on the scoring rule rather than on the model's reasoning.

Note: the strict rule approximates FinBen's metric; it isn't FinBen's exact evaluation code.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest

import finqa_eval as fe

MODELS = ["qwen3.5:9b", "gemma4:26b", "qwen3-coder:30b"]
SETTINGS = ["cot", "tool"]
SETTING_LABEL = {"cot": "Chain-of-thought", "tool": "Tool use (Python)"}
RESULTS_DIR = Path("results")
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
TAB_DIR = Path("tables"); TAB_DIR.mkdir(exist_ok=True)

# Chart style: thin bars, recessive grid, no top/right frame
COLORS = {"cot": "#2a78d6", "tool": "#eb6834", "finben": "#a3a29c"}
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False,
})

## Load and score

In [ ]:
items = {it["id"]: it for it in fe.load_finqa("data")}

records = []
for path in sorted(RESULTS_DIR.glob("*.jsonl")):
    records += [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
df = pd.DataFrame(records)

df["strict"] = [fe.score_strict(f, items[i]) for f, i in zip(df["final"], df["id"])]
df["tolerant"] = [fe.score_tolerant(f, items[i]) for f, i in zip(df["final"], df["id"])]
df["no_answer"] = df["final"].isna()
df["gold_answer"] = df["id"].map(lambda i: items[i]["gold_answer"])
df["gold_exe"] = df["id"].map(lambda i: items[i]["gold_exe"])
df["question"] = df["id"].map(lambda i: items[i]["question"])
df["model"] = pd.Categorical(df["model"], MODELS, ordered=True)
df["setting"] = pd.Categorical(df["setting"], SETTINGS, ordered=True)

df.groupby(["model", "setting"], observed=True).size().rename("n questions").to_frame()

## Table 1: Accuracy with 95% bootstrap confidence intervals

In [ ]:
def bootstrap_ci(x, n_boot=5000, seed=0):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x))).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

rows = []
for (model, setting), g in df.groupby(["model", "setting"], observed=True):
    row = {"model": model, "setting": setting, "n": len(g)}
    for metric in ["strict", "tolerant"]:
        lo, hi = bootstrap_ci(g[metric])
        row[metric] = g[metric].mean()
        row[f"{metric}_lo"], row[f"{metric}_hi"] = lo, hi
    row["gap (tol − strict)"] = row["tolerant"] - row["strict"]
    row["no answer"] = g["no_answer"].mean()
    row["s/question"] = g["seconds"].mean()
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(TAB_DIR / "table1_accuracy.csv", index=False)

show = summary.copy()
for m in ["strict", "tolerant"]:
    show[m] = [f"{v:.2f} [{lo:.2f}, {hi:.2f}]" for v, lo, hi in zip(show[m], show[f"{m}_lo"], show[f"{m}_hi"])]
show[["model", "setting", "n", "strict", "tolerant", "gap (tol − strict)", "no answer", "s/question"]].round(2)

## Figure 1: Accuracy by model and setting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
x = np.arange(len(MODELS)); w = 0.36
for ax, metric, title in zip(axes, ["strict", "tolerant"], ["Strict exact match (FinBen-style)", "Tolerant numeric match"]):
    for k, setting in enumerate(SETTINGS):
        s = summary[summary["setting"] == setting].set_index("model").reindex(MODELS)
        pos = x + (k - 0.5) * (w + 0.02)
        ax.bar(pos, s[metric], w, color=COLORS[setting], label=SETTING_LABEL[setting])
        ax.errorbar(pos, s[metric], yerr=[s[metric] - s[f"{metric}_lo"], s[f"{metric}_hi"] - s[metric]],
                    fmt="none", ecolor="#52514e", elinewidth=1, capsize=3)
        for p, v, hi in zip(pos, s[metric], s[f"{metric}_hi"]):
            ax.text(p, hi + 0.02, f"{v:.2f}", ha="center", fontsize=8, color="#52514e")
    ax.set_xticks(x, MODELS); ax.set_title(title, fontsize=10, loc="left"); ax.set_ylim(0, 1)
axes[0].set_ylabel("Accuracy (200 FinQA questions)")
axes[0].legend(loc="upper left")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig1_accuracy.pdf"); plt.show()

## Figure 2: Comparison with FinBen (Table 3 of Xie et al., 2024)

FinBen's FinQA results (EmAcc, zero-shot) next to ours under the strict rule. Values marked * in FinBen were taken from earlier papers.
Caveat: different samples (FinBen: all 1,147 questions; here: 200) and a different prompt and harness. The comparison is indicative, not exact.

In [ ]:
finben = pd.Series({
    "GPT-4*": 0.63, "ChatGPT*": 0.58, "LLaMA2-70B": 0.06, "FinMA-7B": 0.04, "Gemini": 0.00,
    "LLaMA2-7B-chat": 0.00, "LLaMA3-8B": 0.00, "FinGPT-7B": 0.00, "InternLM-7B": 0.00,
    "Falcon-7B": 0.00, "Mixtral-7B": 0.00, "CFGPT-7B": 0.00,
}, name="FinQA EmAcc")

ours = summary.set_index(["model", "setting"])["strict"]
comp = pd.concat([
    finben.rename("accuracy").to_frame().assign(source="FinBen (2024)"),
    pd.DataFrame({"accuracy": [ours[(m, s)] for m in MODELS for s in SETTINGS if (m, s) in ours.index],
                  "source": [f"this study ({s})" for m in MODELS for s in SETTINGS if (m, s) in ours.index]},
                 index=[f"{m} ({s})" for m in MODELS for s in SETTINGS if (m, s) in ours.index]),
]).sort_values("accuracy")
comp.to_csv(TAB_DIR / "table2_finben_comparison.csv")

fig, ax = plt.subplots(figsize=(7, 5.5))
colors = [COLORS["finben"] if src.startswith("FinBen") else COLORS["tool" if "tool" in src else "cot"] for src in comp["source"]]
ax.barh(comp.index, comp["accuracy"], color=colors, height=0.7)
for i, v in enumerate(comp["accuracy"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
ax.set_xlim(0, 1); ax.set_xlabel("FinQA accuracy (strict exact match)")
ax.grid(axis="x", color="#e6e5e0"); ax.grid(axis="y", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COLORS["finben"], label="FinBen (2024), earlier LLMs"),
                   Patch(color=COLORS["cot"], label="This study: chain-of-thought"),
                   Patch(color=COLORS["tool"], label="This study: tool use")], loc="lower right")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_finben_comparison.pdf"); plt.show()

## Table 3: Paired tests (McNemar, exact)

Every model and setting answered the *same* 200 questions, so comparisons are paired: only the questions where exactly one of the two is correct matter. Tolerant scoring.

In [ ]:
def mcnemar(a, b):
    # a, b: boolean Series indexed by question id (same questions)
    a_only, b_only = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return a_only, b_only, p

wide = df.pivot_table(index="id", columns=["model", "setting"], values="tolerant", observed=True).astype(bool)
tests = []
for m in MODELS:                       # tool vs. chain-of-thought, per model
    if (m, "tool") in wide and (m, "cot") in wide:
        t, c, p = mcnemar(wide[(m, "tool")], wide[(m, "cot")])
        tests.append({"comparison": f"{m}: tool vs. cot", "only first correct": t, "only second correct": c, "p-value": p})
for s in SETTINGS:                     # model vs. model, per setting
    for i, m1 in enumerate(MODELS):
        for m2 in MODELS[i + 1:]:
            if (m1, s) in wide and (m2, s) in wide:
                a, b, p = mcnemar(wide[(m1, s)], wide[(m2, s)])
                tests.append({"comparison": f"{s}: {m1} vs. {m2}", "only first correct": a, "only second correct": b, "p-value": p})
tests = pd.DataFrame(tests)
tests.to_csv(TAB_DIR / "table3_mcnemar.csv", index=False)
tests.round(4)

## Tool setting: how often did the code run, and did the repair round help?

In [ ]:
tool = df[df["setting"] == "tool"]
tool_stats = tool.groupby("model", observed=True).agg(
    n=("id", "size"),
    first_try_ok=("n_repairs", lambda r: (r == 0).mean()),
    needed_repair=("n_repairs", "mean"),
    failed_after_repair=("exec_error", lambda e: e.notna().mean()),
    accuracy_tolerant=("tolerant", "mean"),
)
rep = tool[tool["n_repairs"] == 1]
tool_stats["repair_success_rate"] = rep.groupby("model", observed=True)["exec_error"].apply(lambda e: e.isna().mean())
tool_stats.round(2)

## Speed

In [ ]:
speed = df.groupby(["model", "setting"], observed=True).agg(
    seconds_mean=("seconds", "mean"), seconds_median=("seconds", "median"),
    output_tokens_mean=("output_tokens", "mean"), prompt_tokens_mean=("prompt_tokens", "mean"),
    truncated_prompts=("truncated_prompt", "sum"),
)
speed.round(1)

## Error analysis

1. The cell below exports 30 random wrong answers (tolerant scoring) per setting to `tables/error_sample.csv`.
2. Open the file and fill in the `category` column by hand, with one of: `extraction` (wrong number taken from table or text), `reasoning` (wrong formula or steps), `arithmetic` (right formula, wrong calculation), `scale/sign` (unit, %, sign or magnitude), `format` (no or unparseable answer), `gold` (the gold answer itself looks wrong).
3. Save it as `tables/error_sample_labeled.csv` and run the following cell to get the figure.

The export only runs if the labeled file doesn't exist yet, so your labels are never overwritten.

In [ ]:
labeled = TAB_DIR / "error_sample_labeled.csv"
if not labeled.exists():
    wrong = df[~df["tolerant"]]
    err = pd.concat([wrong[wrong["setting"] == s].pipe(lambda g: g.sample(min(30, len(g)), random_state=0))
                     for s in SETTINGS])
    err = err[["id", "model", "setting", "question", "gold_answer", "gold_exe", "final", "response", "code"]].assign(category="")
    err.to_csv(TAB_DIR / "error_sample.csv", index=False)
    print(f"Exported {len(err)} errors to tables/error_sample.csv – label them, save as error_sample_labeled.csv")

In [ ]:
if labeled.exists():
    lab = pd.read_csv(labeled)
    counts = pd.crosstab(lab["category"], lab["setting"]).reindex(columns=SETTINGS, fill_value=0)
    counts = counts.loc[counts.sum(axis=1).sort_values().index]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    y = np.arange(len(counts)); h = 0.38
    for k, s in enumerate(SETTINGS):
        ax.barh(y + (k - 0.5) * (h + 0.02), counts[s], h, color=COLORS[s], label=SETTING_LABEL[s])
    ax.set_yticks(y, counts.index); ax.set_xlabel("Number of errors (hand-labeled sample)")
    ax.grid(axis="x", color="#e6e5e0"); ax.grid(axis="y", visible=False); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_error_categories.pdf"); plt.show()
    display(counts)
else:
    print("No labeled file yet.")

## Examples for the paper

One question that tool use solved but chain-of-thought didn't (and the other way round): useful as illustrations.

In [ ]:
m = MODELS[-1]
if (m, "tool") in wide and (m, "cot") in wide:
    for label, mask in [("tool right, cot wrong", wide[(m, "tool")] & ~wide[(m, "cot")]),
                        ("cot right, tool wrong", ~wide[(m, "tool")] & wide[(m, "cot")])]:
        ids = mask[mask].index
        if len(ids):
            qid = ids[0]
            print(f"=== {label} ({m}) — {qid}\nQ: {items[qid]['question']}\nGold: {items[qid]['gold_answer']} ({items[qid]['gold_exe']})")
            for s in SETTINGS:
                r = df[(df["id"] == qid) & (df["model"] == m) & (df["setting"] == s)].iloc[0]
                print(f"--- {s}: final = {r['final']}\n{(r['code'] if s == 'tool' else r['response'])[-700:]}")
            print()